In [1]:
from nns.memorygraphs.graph_memorypool import Config, GraphIIConfig, MultilevelCoordinator

cfg = Config()
g2cfg = GraphIIConfig()
cfg.H = 512; cfg.W = 512
mc = MultilevelCoordinator(cfg, g2cfg)

from nns.cnns.features import RetinaModel, MultiScaleFeatureBank
retina = RetinaModel(edge_apply_gaussian=True, edge_gauss_kernel_size=5, edge_gauss_sigma=1.0)
msbank = MultiScaleFeatureBank(scales=[1.0, 2.0, 4.0], base_sigma=1.0)

In [2]:
from trail.pic_tools import *
from trail.matrix_tools import *
# import random
# from trail.foveated_visualization1 import visualize_step
import glob
import os

# folder pic

In [ ]:
folder_path = '/home/p/code/ILSVRC/Data/DET/train/ILSVRC2014_train_0006'

# image_paths = glob.glob(os.path.join(folder_path, '*.JPEG'))

image_paths = glob.iglob(folder_path + '/*.JPEG')
print(image_paths)
pic = 1
for image_path in image_paths:
    print(f"Picture {pic}")
    pic += 1
    
    tensor, oringinal_image = image_to_tensor(image_path)
    B, C, W, H = tensor.shape

    # retina_out = retina(tensor, center_x = -1, center_y = -1)
    _, grad, hue, _, _, cropped = retina(tensor, center_x = -1, center_y = -1)
    # msfb_out = msbank(cropped)
    cur, asp, ori = msbank(cropped)


    step = 1
    mc.handle_new_view({'grad': grad, 'hue': hue, 'curvature': cur, 'aspect': asp, 'orientation': ori})
    step += 1

    while step < 200:
        mc.handle_new_view({'grad': grad, 'hue': hue, 'curvature': cur, 'aspect': asp, 'orientation': ori})
        step += 1

    print(f"Step {step}, graph1 nodes in total = {mc.graphI.graphI[0]._next_proto_id} ")
    print(f"Step {step}, graph2 nodes in total = {mc.graphII.next_graph2_id} ")

    mc.graphI.reset_all_buckets()
    # if 5 == pic:
    #     break



<generator object _iglob at 0x7efe41728cc0>
Picture 1


AttributeError: 'MultilevelCoordinator' object has no attribute 'GraphI'

# single pic

In [ ]:
image_path = 'picture/OIP-C.jpg' 
# image_path = "d:/code/imgs/ILSVRClarge/Data/DET/train/ILSVRC2014_train_0003/ILSVRC2014_train_00030158.JPEG"
tensor, oringinal_image = image_to_tensor(image_path)
B, C, W, H = tensor.shape

# retina_out = retina(tensor, center_x = -1, center_y = -1)
_, grad, hue, _, _, cropped = retina(tensor, center_x = -1, center_y = -1)
# msfb_out = msbank(cropped)
cur, asp, ori = msbank(cropped)

In [ ]:
i = 1
mode, Nactivated_nodes, Nactive_protos, center = fm.run_one_step(grad, hue, cur, asp, ori, reset_image=True)
print(f"Step {i}: mode={mode}, center={center}, #Nactivated_nodes={Nactivated_nodes}, #Nactive_protos:={Nactive_protos}")
print("mode_next_step", fm.graphI.saccade.mode)
print("fm.graphII.next_graph2_id ", fm.graphII.next_graph2_id)
i += 1

while mode != 'end' and i < 200:
    mode, Nactivated_nodes, Nactive_protos, center = fm.run_one_step(grad, hue, cur, asp, ori, reset_image=False)
    print(f"Step {i}: mode={mode}, center={center}, #Nactivated_nodes={Nactivated_nodes}, #Nactive_protos:={Nactive_protos}")
    print("mode_next_step", fm.graphI.saccade.mode)
    print("fm.graphII.next_graph2_id ", fm.graphII.next_graph2_id)
    i += 1

# trail tools

In [ ]:
mode, Nactivated_nodes, Nactive_protos, center = fm.run_one_step(grad, hue, cur, asp, ori, reset_image=True)

print("Step 1 mode:", mode, ", center:", center, ", #Nactivated_nodes:", Nactivated_nodes, ", #Nactive_protos:", Nactive_protos)
print(fm.graphI.saccade.mode)

In [ ]:
i = 2
while mode != 'end' and i < 200:
    mode, Nactivated_nodes, Nactive_protos, center = fm.run_one_step(grad, hue, cur, asp, ori, reset_image=False)
    print(f"Step {i}: mode={mode}, center={center}, #Nactivated_nodes={Nactivated_nodes}, #Nactive_protos:={Nactive_protos}")
    print(fm.graphI.saccade.mode)
    i += 1


In [ ]:
print(len(fm.graphII.graph2_nodes))
print(len(fm.graphII.graph2_edges))
print(fm.graphII.next_graph2_id)

In [ ]:

print('0', fm.graphI.graphI[0]._next_node_id, fm.graphI.graphI[0]._next_proto_id)
print('1', fm.graphI.graphI[1]._next_node_id, fm.graphI.graphI[1]._next_proto_id)
print('2', fm.graphI.graphI[2]._next_node_id, fm.graphI.graphI[2]._next_proto_id)
print('3', fm.graphI.graphI[3]._next_node_id, fm.graphI.graphI[3]._next_proto_id)
print('4', fm.graphI.graphI[4]._next_node_id, fm.graphI.graphI[4]._next_proto_id)